In [1]:
from google.colab import files
uploaded = files.upload()

Saving level2_realworld_dataset_12000.csv to level2_realworld_dataset_12000.csv


In [2]:
import pandas as pd

df = pd.read_csv("level2_realworld_dataset_12000.csv")
df.head()

,text,iodine,vitamin_e,calcium,iron,riboflavin_b2,folate_b9,vitamin_c,vitamin_b6,vitamin_d,zinc
0,cracked lips sore throat no prior illness repo...,0,0,0,0,1,0,0,0,0,0
1,weight gain hormone imbalance fatigue goiter t...,1,0,0,1,0,0,0,0,0,0
2,confusion depression routine checkup,0,0,0,0,0,0,0,1,0,0
3,fatigue thyroid swelling hormone imbalance wei...,1,1,0,0,1,0,0,0,0,0
4,skin rash cracked lips sore throat depression ...,0,0,0,0,1,0,0,1,0,0


In [3]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   text           12000 non-null  object
 1   iodine         12000 non-null  int64 
 2   vitamin_e      12000 non-null  int64 
 3   calcium        12000 non-null  int64 
 4   iron           12000 non-null  int64 
 5   riboflavin_b2  12000 non-null  int64 
 6   folate_b9      12000 non-null  int64 
 7   vitamin_c      12000 non-null  int64 
 8   vitamin_b6     12000 non-null  int64 
 9   vitamin_d      12000 non-null  int64 
 10  zinc           12000 non-null  int64 
dtypes: int64(10), object(1)
memory usage: 1.0+ MB


,0
text,0
iodine,0
vitamin_e,0
calcium,0
iron,0
riboflavin_b2,0
folate_b9,0
vitamin_c,0
vitamin_b6,0
vitamin_d,0


In [4]:
X = df['text']

y = df[['iodine','vitamin_e','calcium','iron',
        'riboflavin_b2','folate_b9','vitamin_c',
        'vitamin_b6','vitamin_d','zinc']]

In [5]:
X = X.str.lower()

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,3),     # 🔥 bigrams + trigrams
    stop_words='english',
    min_df=2
)

X = vectorizer.fit_transform(X)

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

In [8]:
from sklearn.svm import LinearSVC
from sklearn.multioutput import MultiOutputClassifier

model = MultiOutputClassifier(
    LinearSVC(class_weight='balanced')   # 🔥 handles weak classes
)

model.fit(X_train, y_train)

MultiOutputClassifier(estimator=LinearSVC(class_weight='balanced'))

In [9]:
from sklearn.metrics import classification_report

pred = model.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.81      0.95      0.87       390
           1       0.81      0.95      0.87       385
           2       0.77      0.97      0.85       378
           3       0.81      0.92      0.86       378
           4       0.83      0.95      0.88       393
           5       0.84      0.94      0.89       390
           6       0.76      0.94      0.84       375
           7       0.81      0.97      0.88       396
           8       0.81      0.92      0.86       410
           9       0.76      0.96      0.85       362

   micro avg       0.80      0.95      0.87      3857
   macro avg       0.80      0.95      0.87      3857
weighted avg       0.80      0.95      0.87      3857
 samples avg       0.79      0.87      0.81      3857



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [10]:
from sklearn.metrics import accuracy_score, hamming_loss

print("Subset Accuracy:", accuracy_score(y_test, pred))
print("Hamming Accuracy:", 1 - hamming_loss(y_test, pred))

Subset Accuracy: 0.60375
Hamming Accuracy: 0.9534583333333333


In [11]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)

print("CV Scores:", scores)
print("Average:", scores.mean())

CV Scores: [0.60791667 0.62083333 0.58083333 0.56666667 0.60083333]
Average: 0.5954166666666666


In [12]:
sample = ["no fatigue but severe bone pain and low sunlight exposure"]

sample_vec = vectorizer.transform(sample)

prediction = model.predict(sample_vec)

print(prediction)

[[0 0 0 0 0 0 0 0 1 0]]


In [13]:
from sklearn.model_selection import cross_val_score

# F1 (micro)
f1_scores = cross_val_score(model, X, y, cv=5, scoring='f1_micro')

print("F1 CV Scores:", f1_scores)
print("Average F1:", f1_scores.mean())

F1 CV Scores: [0.86780384 0.87160085 0.85966377 0.85608171 0.86654867]
Average F1: 0.8643397671170145


In [14]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, hamming_loss

# Hamming loss scorer
hamming_scorer = make_scorer(hamming_loss, greater_is_better=False)

hamming_scores = cross_val_score(model, X, y, cv=5, scoring=hamming_scorer)

# Convert loss → accuracy
hamming_accuracy = 1 - (-hamming_scores.mean())

print("Hamming CV Scores:", -hamming_scores)
print("Average Hamming Accuracy:", hamming_accuracy)

Hamming CV Scores: [0.0465     0.04525    0.04904167 0.05166667 0.047125  ]
Average Hamming Accuracy: 0.9520833333333334


In [15]:
print(len(df['text'].unique()))

12000


In [16]:
print("Total rows:", len(df))
print("Unique texts:", len(df['text'].unique()))

Total rows: 12000
Unique texts: 12000


In [17]:
duplicates = len(df) - len(df['text'].unique())
print("Duplicate texts:", duplicates)

Duplicate texts: 0


In [18]:
import joblib

joblib.dump(model, "model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

['vectorizer.pkl']

In [19]:
from google.colab import files

files.download("model.pkl")
files.download("vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>